# 04 — Pairs and Correlation Analysis

Cointegration scanning, spread visualization, and cluster-based relative value.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


In [ ]:
from statarb.signals.pairs import PairsSignals
from statarb.signals.themes import CRYPTO_THEMES
ps = PairsSignals()
print("Available thematic clusters:")
for theme, members in CRYPTO_THEMES.items():
    present = [m for m in members if m in returns.columns]
    print(f"  {theme:20s}: {present}")


## Correlation Clustering

In [ ]:
# Compute correlation matrix and cluster assets
from scipy.cluster.hierarchy import dendrogram, linkage

corr = returns.dropna().corr()
# Convert correlation to distance
dist = np.sqrt(2 * (1 - corr.clip(-1, 1)))

fig, ax = plt.subplots(figsize=(12, 5))
linked = linkage(dist.values[np.triu_indices(len(dist), k=1)], method="ward")
dendrogram(linked, labels=list(corr.columns), leaf_rotation=45,
           leaf_font_size=8, ax=ax)
ax.set_title("Asset Clustering by Return Correlation (Ward Linkage)", fontweight="bold")
plt.tight_layout()
plt.show()


## Cointegration Scan

In [ ]:
# Find cointegrated pairs (Engle-Granger)
print("Scanning for cointegrated pairs...")
try:
    lookback = min(252, len(prices) - 10)
    coint_pairs = ps.find_cointegrated_pairs(prices, lookback=lookback, pvalue_threshold=0.10)
    print(f"Found {len(coint_pairs)} cointegrated pairs (p < 0.10)")
    if coint_pairs:
        print("\nTop pairs:")
        for a, b, pval in coint_pairs[:10]:
            print(f"  {a:12s} / {b:12s}  p={pval:.4f}")
except Exception as e:
    print(f"Cointegration scan unavailable: {e}")
    coint_pairs = []


## Spread Visualization

In [ ]:
# Visualize spread z-score for available asset pairs
available_cols = list(prices.columns)
plot_pairs = []
for i, a in enumerate(available_cols[:5]):
    for b in available_cols[i+1:6]:
        plot_pairs.append((a, b))

n_plots = min(4, len(plot_pairs))
if n_plots > 0:
    fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3 * n_plots))
    if n_plots == 1:
        axes = [axes]
    for ax, (a, b) in zip(axes, plot_pairs[:n_plots]):
        try:
            spread = ps.spread_zscore(prices[a], prices[b], window=21)
            ax.plot(spread.index, spread.values, linewidth=0.8, color="navy")
            ax.axhline(0, color="black", linewidth=0.8)
            ax.axhline(1.5, color="red", linestyle="--", alpha=0.7, linewidth=0.8)
            ax.axhline(-1.5, color="green", linestyle="--", alpha=0.7, linewidth=0.8)
            ax.set_title(f"{a.split('/')[0]} / {b.split('/')[0]} Spread Z-Score (21-day)",
                        fontsize=9)
            ax.set_ylabel("Z-Score")
        except Exception as e:
            ax.set_title(f"Error: {e}")
    plt.tight_layout()
    plt.show()


## Cluster Momentum

In [ ]:
# Cluster-based momentum: buy assets in outperforming themes
try:
    cluster_mom = ps.cluster_momentum_signal(returns, CRYPTO_THEMES, lookback=21)
    rel_val = ps.relative_value_signal(returns, correlation_window=63,
                                        return_window=21, min_correlation=0.5)
    signals = {
        "cluster_momentum": fe.cross_sectional_rank(cluster_mom),
        "relative_value":   fe.cross_sectional_rank(rel_val),
    }
    from experiments._utils import backtest_signals
    from statarb.backtest.execution import ExecutionModel
    results = backtest_signals(signals, returns, ExecutionModel(), 365)
    print(results[["sharpe", "annualized_return", "max_drawdown"]].round(3).to_string())
except Exception as e:
    print(f"Signal construction failed (may need more assets): {e}")
